# Original Dataset Investigation

Kaggle states that the competition dataset was inspired by the
EV Adoption Behavior dataset.

Goal:

1. Load the original 10,000-row dataset.
2. Compare its schema and distributions with the competition data.
3. Check missing values and exact overlap.
4. Determine whether the original data can safely improve training.
5. Validate any augmentation only on untouched competition rows.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
import sys
import subprocess

subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        sys.executable,
        "kagglehub"
    ],
    check=True
)

Using Python 3.12.14 environment at: /home/zulkifil/projects/ev-purchase-prediction/.venv
Resolved 11 packages in 2.08s
Prepared 3 packages in 184ms
Installed 3 packages in 13ms
 + kagglehub==1.0.2
 + kagglesdk==0.1.37
 + protobuf==7.36.1


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/home/zulkifil/projects/ev-purchase-prediction/.venv/bin/python', 'kagglehub'], returncode=0)

In [3]:
import kagglehub

/home/zulkifil/projects/ev-purchase-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
original_dir = Path("../data/original")
original_dir.mkdir(
    parents=True,
    exist_ok=True
)

original_path = kagglehub.dataset_download(
    "itzzomkar/ev-adoption-behavior-and-range-anxiety",
    path="EV_Adoption_and_Range_Anxiety_Dataset.csv",
    output_dir=str(original_dir)
)

print(original_path)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 676k/676k [00:01<00:00, 433kB/s]

../data/original/EV_Adoption_and_Range_Anxiety_Dataset.csv


In [5]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

original = pd.read_csv(original_path)

print("Competition train:", train.shape)
print("Competition test: ", test.shape)
print("Original dataset: ", original.shape)

Competition train: (668665, 15)
Competition test:  (286571, 14)
Original dataset:  (10000, 15)


In [6]:
print("Competition:")
print(train.columns.tolist())

print("\nOriginal:")
print(original.columns.tolist())

Competition:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']

Original:
['Buyer_ID', 'Age', 'Gender', 'Annual_Income_USD', 'City_Type', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Current_Car_Type', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Home_Charging_Possible', 'Environmental_Concern_Level', 'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']


In [7]:
competition_features = set(
    train.columns
) - {
    "id",
    "Will_Buy_EV"
}

original_features = set(
    original.columns
) - {
    "Buyer_ID",
    "Will_Buy_EV"
}

print(
    "Only in competition:",
    competition_features - original_features
)

print(
    "Only in original:",
    original_features - competition_features
)

Only in competition: set()
Only in original: set()


In [8]:
original.head()

,Buyer_ID,Age,Gender,Annual_Income_USD,City_Type,Daily_Commute_km,Number_of_Cars_Owned,Current_Car_Type,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Home_Charging_Possible,Environmental_Concern_Level,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,EV00001,56,Male,44906.0,Rural,18.1,1,Truck,1,0,Yes,4.0,Yes,Low,No
1,EV00002,36,Male,88754.0,Rural,5.0,1,SUV,2,3,Yes,1.0,Yes,Low,No
2,EV00003,42,Male,155958.0,Urban,32.2,2,Sedan,14,3,No,2.0,Yes,Medium,Yes
3,EV00004,31,Male,37059.0,Rural,89.9,1,Sedan,2,1,Yes,4.0,No,Low,No
4,EV00005,48,Female,48552.0,Rural,71.0,1,Sedan,2,0,Yes,3.0,Yes,Low,No


In [9]:
original.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Buyer_ID                     10000 non-null  str    
 1   Age                          10000 non-null  int64  
 2   Gender                       10000 non-null  str    
 3   Annual_Income_USD            9822 non-null   float64
 4   City_Type                    10000 non-null  str    
 5   Daily_Commute_km             9819 non-null   float64
 6   Number_of_Cars_Owned         10000 non-null  int64  
 7   Current_Car_Type             10000 non-null  str    
 8   Charging_Stations_Near_Home  10000 non-null  int64  
 9   Charging_Stations_Near_Work  10000 non-null  int64  
 10  Home_Charging_Possible       10000 non-null  str    
 11  Environmental_Concern_Level  9816 non-null   float64
 12  Subsidy_Available            10000 non-null  str    
 13  Range_Anxiety_Level         

In [10]:
original.isna().sum().sort_values(
    ascending=False
)

Environmental_Concern_Level    184
Daily_Commute_km               181
Annual_Income_USD              178
Gender                           0
Age                              0
City_Type                        0
Buyer_ID                         0
Number_of_Cars_Owned             0
Current_Car_Type                 0
Charging_Stations_Near_Work      0
Charging_Stations_Near_Home      0
Home_Charging_Possible           0
Subsidy_Available                0
Range_Anxiety_Level              0
Will_Buy_EV                      0
dtype: int64

In [11]:
original["Will_Buy_EV"].value_counts()

Will_Buy_EV
No     8250
Yes    1750
Name: count, dtype: int64

In [12]:
original[
    "Will_Buy_EV"
].value_counts(
    normalize=True
)

Will_Buy_EV
No     0.825
Yes    0.175
Name: proportion, dtype: float64

In [13]:
train[
    "Will_Buy_EV"
].value_counts(
    normalize=True
)

Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64

In [14]:
numeric_features = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level"
]

In [15]:
comparison_rows = []

for col in numeric_features:

    comparison_rows.append({
        "Feature": col,

        "Competition_Mean":
            train[col].mean(),

        "Original_Mean":
            original[col].mean(),

        "Competition_Std":
            train[col].std(),

        "Original_Std":
            original[col].std()
    })

numeric_comparison = pd.DataFrame(
    comparison_rows
)

numeric_comparison

,Feature,Competition_Mean,Original_Mean,Competition_Std,Original_Std
0,Age,47.039171,46.936200,12.875448,12.945968
1,Annual_Income_USD,84769.266989,85378.485848,28648.029042,32838.684167
2,Daily_Commute_km,32.158298,41.105418,18.730474,23.351276
3,Number_of_Cars_Owned,1.712626,1.859900,0.729275,0.855421
4,Charging_Stations_Near_Home,4.960408,5.346900,3.926843,4.049074
5,Charging_Stations_Near_Work,7.176314,7.458200,5.186627,5.259374
6,Environmental_Concern_Level,2.935477,2.982783,1.429119,1.413784


In [16]:
categorical_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

In [17]:
for col in categorical_features:

    print("=" * 50)
    print(col)

    print(
        "Competition:",
        sorted(
            train[col]
            .dropna()
            .astype(str)
            .unique()
        )
    )

    print(
        "Original:",
        sorted(
            original[col]
            .dropna()
            .astype(str)
            .unique()
        )
    )

Gender
Competition: ['Female', 'Male', 'Other']
Original: ['Female', 'Male', 'Other']
City_Type
Competition: ['Rural', 'Suburban', 'Urban']
Original: ['Rural', 'Suburban', 'Urban']
Current_Car_Type
Competition: ['Hatchback', 'SUV', 'Sedan', 'Truck']
Original: ['Hatchback', 'SUV', 'Sedan', 'Truck']
Home_Charging_Possible
Competition: ['No', 'Yes']
Original: ['No', 'Yes']
Subsidy_Available
Competition: ['No', 'Yes']
Original: ['No', 'Yes']
Range_Anxiety_Level
Competition: ['High', 'Low', 'Medium']
Original: ['High', 'Low', 'Medium']


In [18]:
def add_features(df):
    df = df.copy()

    df["Income_per_Car"] = (
        df["Annual_Income_USD"]
        / df["Number_of_Cars_Owned"].clip(lower=1)
    )

    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )

    df["Charging_Station_Difference"] = (
        df["Charging_Stations_Near_Home"]
        - df["Charging_Stations_Near_Work"]
    )

    df["Income_x_Concern"] = (
        df["Annual_Income_USD"]
        * df["Environmental_Concern_Level"]
    )

    df["Commute_x_Concern"] = (
        df["Daily_Commute_km"]
        * df["Environmental_Concern_Level"]
    )

    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    df["Subsidy_x_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Concern_x_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    income_band = pd.cut(
        df["Annual_Income_USD"],
        bins=[
            -np.inf,
            60000,
            80000,
            100000,
            120000,
            np.inf
        ],
        labels=[
            "Income_1",
            "Income_2",
            "Income_3",
            "Income_4",
            "Income_5"
        ]
    ).astype(str)

    df["Income_x_Subsidy"] = (
        income_band
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    commute_band = pd.cut(
        df["Daily_Commute_km"],
        bins=[
            -np.inf,
            20,
            40,
            60,
            np.inf
        ],
        labels=[
            "Short",
            "Medium",
            "Long",
            "VeryLong"
        ]
    ).astype(str)

    df["HomeCharge_x_Commute"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + commute_band
    )

    return df

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

In [20]:
y_comp = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y_comp
)

In [21]:
len(train_idx), len(valid_idx)

(534932, 133733)

In [22]:
feature_columns = [
    col
    for col in train.columns
    if col not in ["id", "Will_Buy_EV"]
]

feature_columns

['Age',
 'Annual_Income_USD',
 'Daily_Commute_km',
 'Number_of_Cars_Owned',
 'Charging_Stations_Near_Home',
 'Charging_Stations_Near_Work',
 'Environmental_Concern_Level',
 'Gender',
 'City_Type',
 'Current_Car_Type',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level']

In [23]:
original_clean = original[
    feature_columns + ["Will_Buy_EV"]
].copy()

In [24]:
competition_train_part = train.iloc[train_idx][
    feature_columns + ["Will_Buy_EV"]
].copy()

In [25]:
competition_valid_part = train.iloc[valid_idx][
    feature_columns + ["Will_Buy_EV"]
].copy()

In [26]:
augmented_train = pd.concat(
    [
        competition_train_part,
        original_clean
    ],
    ignore_index=True
)

print(
    "Competition training rows:",
    len(competition_train_part)
)

print(
    "Original rows:",
    len(original_clean)
)

print(
    "Augmented training rows:",
    len(augmented_train)
)

print(
    "Validation rows:",
    len(competition_valid_part)
)

Competition training rows: 534932
Original rows: 10000
Augmented training rows: 544932
Validation rows: 133733


In [27]:
print(
    "Competition training positive rate:",
    (
        competition_train_part["Will_Buy_EV"]
        == "Yes"
    ).mean()
)

print(
    "Original positive rate:",
    (
        original_clean["Will_Buy_EV"]
        == "Yes"
    ).mean()
)

print(
    "Augmented positive rate:",
    (
        augmented_train["Will_Buy_EV"]
        == "Yes"
    ).mean()
)

Competition training positive rate: 0.17464462772838416
Original positive rate: 0.175
Augmented positive rate: 0.1746511491342039


In [28]:
augmented_train_fe = add_features(
    augmented_train
)

competition_valid_fe = add_features(
    competition_valid_part
)

In [29]:
y_augmented = augmented_train_fe[
    "Will_Buy_EV"
].map({
    "No": 0,
    "Yes": 1
})

y_valid_original_test = competition_valid_fe[
    "Will_Buy_EV"
].map({
    "No": 0,
    "Yes": 1
})

In [30]:
X_augmented = augmented_train_fe.drop(
    columns=["Will_Buy_EV"]
)

X_valid_original_test = competition_valid_fe.drop(
    columns=["Will_Buy_EV"]
)

X_augmented.shape, X_valid_original_test.shape

((544932, 24), (133733, 24))

In [31]:
categorical_cols = X_augmented.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

categorical_cols

['Gender',
 'City_Type',
 'Current_Car_Type',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level',
 'Subsidy_x_Concern',
 'Subsidy_x_Anxiety',
 'Concern_x_Anxiety',
 'Income_x_Subsidy',
 'HomeCharge_x_Subsidy',
 'HomeCharge_x_Commute']

In [33]:
X_augmented = augmented_train_fe.drop(
    columns=["Will_Buy_EV"]
).copy()

X_valid_original_test = competition_valid_fe.drop(
    columns=["Will_Buy_EV"]
).copy()

X_augmented.shape, X_valid_original_test.shape

((544932, 24), (133733, 24))

In [35]:
categorical_cols = X_augmented.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

categorical_cols

['Gender',
 'City_Type',
 'Current_Car_Type',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level',
 'Subsidy_x_Concern',
 'Subsidy_x_Anxiety',
 'Concern_x_Anxiety',
 'Income_x_Subsidy',
 'HomeCharge_x_Subsidy',
 'HomeCharge_x_Commute']

In [36]:
for col in categorical_cols:

    train_values = (
        X_augmented[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    valid_values = (
        X_valid_original_test[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    categories = pd.concat(
        [
            train_values,
            valid_values
        ],
        ignore_index=True
    ).dropna().unique().tolist()

    X_augmented[col] = pd.Categorical(
        train_values,
        categories=categories
    )

    X_valid_original_test[col] = pd.Categorical(
        valid_values,
        categories=categories
    )

In [37]:
X_augmented[categorical_cols].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
Subsidy_x_Concern         category
Subsidy_x_Anxiety         category
Concern_x_Anxiety         category
Income_x_Subsidy          category
HomeCharge_x_Subsidy      category
HomeCharge_x_Commute      category
dtype: object

In [38]:
X_augmented.isna().sum().sort_values(
    ascending=False
).head(10)

Commute_x_Concern              362
Income_x_Concern               359
Environmental_Concern_Level    184
Daily_Commute_km               181
Annual_Income_USD              178
Income_per_Car                 178
Number_of_Cars_Owned             0
Age                              0
Gender                           0
Charging_Stations_Near_Work      0
dtype: int64

In [41]:
original_aug_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [42]:
original_aug_model.fit(
    X_augmented,
    y_augmented,

    eval_X=X_valid_original_test,
    eval_y=y_valid_original_test,

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.940756	valid_0's binary_logloss: 0.22847
[400]	valid_0's auc: 0.941422	valid_0's binary_logloss: 0.227199
[600]	valid_0's auc: 0.941582	valid_0's binary_logloss: 0.226898
[800]	valid_0's auc: 0.941627	valid_0's binary_logloss: 0.2268
[1000]	valid_0's auc: 0.94167	valid_0's binary_logloss: 0.226723
[1200]	valid_0's auc: 0.941684	valid_0's binary_logloss: 0.226691
Early stopping, best iteration is:
[1153]	valid_0's auc: 0.941698	valid_0's binary_logloss: 0.226668
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [43]:
original_aug_pred = original_aug_model.predict_proba(
    X_valid_original_test
)[:, 1]

original_aug_auc = roc_auc_score(
    y_valid_original_test,
    original_aug_pred
)

In [44]:
print(f"Raw LightGBM:          {0.941669:.6f}")
print(f"Feature Engineering:   {0.941785:.6f}")
print(f"FE + Target Encoding:  {0.941771:.6f}")
print(f"FE + Original Data:    {original_aug_auc:.6f}")

print()
print(
    f"Original-data gain vs FE: "
    f"{original_aug_auc - 0.941785:+.6f}"
)

print(
    f"Best iteration: "
    f"{original_aug_model.best_iteration_}"
)

Raw LightGBM:          0.941669
Feature Engineering:   0.941785
FE + Target Encoding:  0.941771
FE + Original Data:    0.941698

Original-data gain vs FE: -0.000087
Best iteration: 1153


In [45]:
source_numeric_features = [
    "Annual_Income_USD",
    "Environmental_Concern_Level"
]

source_categorical_features = [
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

source_features = (
    source_numeric_features
    + source_categorical_features
)

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import LogisticRegression

In [47]:
source_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            source_numeric_features
        ),

        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            source_categorical_features
        )
    ]
)

In [48]:
source_model = Pipeline([
    (
        "preprocess",
        source_preprocessor
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [49]:
y_original = original[
    "Will_Buy_EV"
].map({
    "No": 0,
    "Yes": 1
})

In [50]:
source_model.fit(
    original[source_features],
    y_original
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['Annual_Income_USD','Environmental_Concern_Level','Subsidy_Available', 'Range_Anxiety_Level']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [51]:
source_valid_pred = source_model.predict_proba(
    competition_valid_part[
        source_features
    ]
)[:, 1]

source_valid_auc = roc_auc_score(
    y_valid_original_test,
    source_valid_pred
)

source_valid_auc

0.9375154046880908

In [52]:
print(
    f"Original-only logistic → "
    f"competition validation AUC: "
    f"{source_valid_auc:.6f}"
)

print(
    f"Competition FE-LightGBM:       "
    f"{0.941785:.6f}"
)

Original-only logistic → competition validation AUC: 0.937515
Competition FE-LightGBM:       0.941785


In [53]:
source_train_score = source_model.decision_function(
    competition_train_part[
        source_features
    ]
)

source_valid_score = source_model.decision_function(
    competition_valid_part[
        source_features
    ]
)

In [54]:
source_train_score[:10]

array([-2.59873196,  1.25639061,  0.28440736, -2.88341274, -2.18424437,
       -4.26595034, -3.8007324 , -4.48819356, -3.81085504, -4.44767903])

In [55]:
competition_train_fe = add_features(
    competition_train_part
)

competition_valid_fe = add_features(
    competition_valid_part
)

In [56]:
X_transfer_train = competition_train_fe.drop(
    columns=["Will_Buy_EV"]
).copy()

X_transfer_valid = competition_valid_fe.drop(
    columns=["Will_Buy_EV"]
).copy()

In [57]:
X_transfer_train[
    "Original_Source_Score"
] = source_train_score

X_transfer_valid[
    "Original_Source_Score"
] = source_valid_score

In [58]:
X_transfer_train.shape, X_transfer_valid.shape

((534932, 25), (133733, 25))

In [59]:
transfer_cat_cols = (
    X_transfer_train
    .select_dtypes(
        include=[
            "object",
            "str",
            "category"
        ]
    )
    .columns
    .tolist()
)

In [60]:
for col in transfer_cat_cols:

    categories = pd.concat(
        [
            X_transfer_train[col]
            .astype(str),

            X_transfer_valid[col]
            .astype(str)
        ],
        ignore_index=True
    ).unique().tolist()

    X_transfer_train[col] = pd.Categorical(
        X_transfer_train[col].astype(str),
        categories=categories
    )

    X_transfer_valid[col] = pd.Categorical(
        X_transfer_valid[col].astype(str),
        categories=categories
    )

In [62]:
transfer_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [63]:
transfer_model.fit(
    X_transfer_train,
    y_comp.iloc[
        train_idx
    ].reset_index(drop=True),

    eval_X=X_transfer_valid,

    eval_y=y_comp.iloc[
        valid_idx
    ].reset_index(drop=True),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.940845	valid_0's binary_logloss: 0.228249
[400]	valid_0's auc: 0.941486	valid_0's binary_logloss: 0.227049
[600]	valid_0's auc: 0.941603	valid_0's binary_logloss: 0.226811
[800]	valid_0's auc: 0.941685	valid_0's binary_logloss: 0.226652
[1000]	valid_0's auc: 0.941712	valid_0's binary_logloss: 0.226605
Early stopping, best iteration is:
[1026]	valid_0's auc: 0.941722	valid_0's binary_logloss: 0.226589
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [64]:
transfer_valid_pred = (
    transfer_model
    .predict_proba(
        X_transfer_valid
    )[:, 1]
)

transfer_auc = roc_auc_score(
    y_comp.iloc[valid_idx],
    transfer_valid_pred
)

transfer_auc

0.9417222704167351

In [65]:
print(
    f"Raw LightGBM:             "
    f"{0.941669:.6f}"
)

print(
    f"Feature Engineering:      "
    f"{0.941785:.6f}"
)

print(
    f"FE + Target Encoding:     "
    f"{0.941771:.6f}"
)

print(
    f"FE + Original Rows:       "
    f"{0.941698:.6f}"
)

print(
    f"Original Logistic Alone:  "
    f"{source_valid_auc:.6f}"
)

print(
    f"FE + Source Score:        "
    f"{transfer_auc:.6f}"
)

print()

print(
    f"Gain vs FE: "
    f"{transfer_auc - 0.941785:+.6f}"
)

print(
    f"Best iteration: "
    f"{transfer_model.best_iteration_}"
)

Raw LightGBM:             0.941669
Feature Engineering:      0.941785
FE + Target Encoding:     0.941771
FE + Original Rows:       0.941698
Original Logistic Alone:  0.937515
FE + Source Score:        0.941722

Gain vs FE: -0.000063
Best iteration: 1026
